### the intention here is to build the workflow where we gather data response from the llm about whats in the pdf guia, that was proven before that it can be accurate, in notebook 04, and then compare with the field DS_PROCEDIMENTO and see if they match or not, generating another df with a aut_validacao_status. we can t keep ds contato , so in the workflow we check if theres a analysis form the analist or not for comparing only purposes of the past data

# Part 1

load the dataframe into memory and bring it to a valid input for unstructured

### DS_CONTATO, meaning the data from query finalizado, we dont care in production stage anymore, so we dont use this base anymore

In [9]:
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.services.database.oracle import OracleService
from app.services.database.mariadb import MariaDBService
from app.utils.db_operations import load_query_from_file, execute_query_to_df
from app.utils.config import load_config
from app.utils.logger import get_logger


logger = get_logger(name=__name__)
config_vars = load_config()

In [10]:
# load the query strings
autorizacao_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_autorizacao.sql"
autorizacao_query_str = load_query_from_file(autorizacao_query_file)

procedimento_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_procedimento.sql"
procedimento_query_str = load_query_from_file(procedimento_query_file)

finalizado_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_pedido_finalizado_resposta.sql"
finalizado_query_str = load_query_from_file(finalizado_query_file)

aviso_cirurgia_query_file = "/home/joao/projects/company_projects/autorizacoes-api/POC/docs/querys/query_aviso_cirurgia.sql"
aviso_cirurgia_query_str = load_query_from_file(aviso_cirurgia_query_file)

In [12]:
maria_db_procedimento_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=procedimento_query_str, fetch_limit=500)
oracle_db_autorizacao_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=autorizacao_query_str)
oracle_db_finalizado_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=finalizado_query_str)
maria_db_aviso_cirurgia_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=aviso_cirurgia_query_str)

{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection established to\n                srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "mariadb.py", "lineno": 27}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "Executing MariaDB query...", "filename": "mariadb.py", "lineno": 67}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ Fetched 280 rows from MariaDB.", "filename": "mariadb.py", "lineno": 75}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection closed.", "filename": "mariadb.py", "lineno": 41}
{"timestamp": "2025-08-27T17:27:49", "level": "INFO", "name": "app.utils.db_operations", "message": "✅ Query executed successfully with 280 sample records", "filename": "db_operations.py", "lineno": 60}


## getting a single table with the info we need

In [14]:
import pandas as pd

maria_db_procedimento_df_processed = maria_db_procedimento_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA", "procedure": "DS_PROCEDIMENTO"})
maria_db_procedimento_df_processed.drop(columns=["hospitalization_type"], inplace=True)
maria_db_procedimento_df_processed = maria_db_procedimento_df_processed[maria_db_procedimento_df_processed["DS_PROCEDIMENTO"].notna()]
maria_db_procedimento_df_processed.reset_index(drop=True, inplace=True)

autorizacao_no_need_cols = [col for col in oracle_db_autorizacao_df.columns if col not in ["CD_AVISO_CIRURGIA", "CD_GUIA", "CD_SENHA", "DS_GUIA_PATH"]]
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df.drop(columns=autorizacao_no_need_cols)
oracle_db_autorizacao_df_processed = oracle_db_autorizacao_df_processed[oracle_db_autorizacao_df_processed["DS_GUIA_PATH"].notna()]
oracle_db_autorizacao_df_processed.reset_index(drop=True, inplace=True)

finalizado_no_need_cols = [col for col in oracle_db_finalizado_df.columns if col not in ["CD_REGISTRO_VINCULADO", "DS_CONTATO"]]
oracle_db_finalizado_df_processed = oracle_db_finalizado_df.drop(columns=finalizado_no_need_cols)
oracle_db_finalizado_df_processed.rename(columns={"CD_REGISTRO_VINCULADO": "CD_AVISO_CIRURGIA"}, inplace=True)
oracle_db_finalizado_df_processed = oracle_db_finalizado_df_processed[oracle_db_finalizado_df_processed["DS_CONTATO"].notna()]
oracle_db_finalizado_df_processed.reset_index(drop=True, inplace=True)

maria_db_aviso_cirurgia_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA"}, inplace=True)
keep_cols = ["CD_AVISO_CIRURGIA", "health_insurance_name"]
maria_db_aviso_cirurgia_df = maria_db_aviso_cirurgia_df[keep_cols]
maria_db_aviso_cirurgia_df.reset_index(drop=True, inplace=True)

In [15]:
merged_df_autorizacao = pd.merge(
    maria_db_procedimento_df_processed,
    oracle_db_autorizacao_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

merged_df_autorizacao_final = pd.merge(
    merged_df_autorizacao,
    oracle_db_finalizado_df_processed,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

final_extracted_df = pd.merge(
    merged_df_autorizacao_final,
    maria_db_aviso_cirurgia_df,
    on="CD_AVISO_CIRURGIA",
    how="inner"
)

In [17]:
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name
0,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE
1,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE
2,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO


In [18]:
# entering the pdf link, downloading the pdf, adding as another col
import requests
import numpy as np
def fetch_pdf_bytes(url):
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200 and 'application/pdf' in response.headers.get('content-type', ''):
            return response.content
        else:
            return np.nan
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return np.nan

# Apply to all links in DS_GUIA_PATH
final_extracted_df['DS_PDF_BYTES'] = final_extracted_df['DS_GUIA_PATH'].apply(fetch_pdf_bytes)
final_extracted_df.head(3)

,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES
0,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE,b'%PDF-1.3\n%\xb7\xbe\xad\xaa\n1 0 obj\n<<\n/T...
1,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE,b'%PDF-1.3\n%\xb7\xbe\xad\xaa\n1 0 obj\n<<\n/T...
2,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...


In [19]:
# Create a new column with clickable links for DS_GUIA_PATH
def make_clickable(url):
    if pd.notna(url):
        return f'<a href="{url}" target="_blank">{url}</a>'
    return ""
final_extracted_df['DS_GUIA_PATH_CLICKABLE'] = final_extracted_df['DS_GUIA_PATH'].apply(make_clickable)
from IPython.display import display, HTML
display(HTML(final_extracted_df[['DS_GUIA_PATH', 'DS_GUIA_PATH_CLICKABLE']].head(10).to_html(escape=False)))

,DS_GUIA_PATH,DS_GUIA_PATH_CLICKABLE
0,https://cdns.overmind.ai/autorizacao-cemig-000120250700053801-1752237175175.pdf,https://cdns.overmind.ai/autorizacao-cemig-000120250700053801-1752237175175.pdf
1,https://cdns.overmind.ai/autorizacao-cemig-000120250700053801-1752237175175.pdf,https://cdns.overmind.ai/autorizacao-cemig-000120250700053801-1752237175175.pdf
2,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223221-1752237144964.pdf
3,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223228-1753454043822.pdf
4,https://cdns.overmind.ai/autorizacao-bradesco-121223237-1753796587472.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121223237-1753796587472.pdf
5,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-203941226-1753068369928.pdf
6,https://cdns.overmind.ai/autorizacao-bradesco-121576086-1753117675678.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121576086-1753117675678.pdf
7,https://cdns.overmind.ai/autorizacao-sulamerica-204656188-1754489468918.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-204656188-1754489468918.pdf
8,https://cdns.overmind.ai/autorizacao-bradesco-121612505-1753983787863.pdf,https://cdns.overmind.ai/autorizacao-bradesco-121612505-1753983787863.pdf
9,https://cdns.overmind.ai/autorizacao-sulamerica-204937675-1754852532865.pdf,https://cdns.overmind.ai/autorizacao-sulamerica-204937675-1754852532865.pdf


# part 2: inputing data to technique, observing the outputed data

In [20]:
# trying to use the mudular code to get the workflow feeling
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.utils.process_images import process_blobs
from app.utils.textract_service import TextractPDFAnalyzer
from app.utils.llm_service import AnthropicLLMService
from app.utils.aws_services_handler import create_boto3_client
from app.utils.config import load_config, AppConstants
from app.utils.logger import get_logger
from app.utils.system_prompts.autorizacao_prompt import Prompts


logger = get_logger(name=__name__)
config_vars = load_config()

In [21]:
app_constants = AppConstants()
textract_client = create_boto3_client("textract", config_vars)
bedrock_client = create_boto3_client("bedrock-runtime", config_vars)
s3_client = create_boto3_client("s3", config_vars)
bucket_name = "autorizacoes"

# Initialize the TextractPDFAnalyzer
pdf_analyzer = TextractPDFAnalyzer(
    textract_client=textract_client,
    s3_client=s3_client,
    bucket_name="autorizacoes-textract",
    bucket_folder="guias-pdf"
)

autorizacao_prompt = Prompts.autorizacao_extraction_prompt


llm_instance = AnthropicLLMService(
    model_id=app_constants.BEDROCK_DEFAULT_MODEL_ID,
    model_version=app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=autorizacao_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)

{"timestamp": "2025-08-27T17:55:39", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-27T17:55:39", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-27T17:55:39", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-27T17:55:39", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-27T17:55:39", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente S3 para a região: us-east-1...

In [26]:

from typing import Dict
# Process rows from the dataframe 
def llm_extraction(df:pd.DataFrame, limit=None) -> Dict:
    if limit:
        df = df[:limit]
    llm_results = {}
    for index, row in df.iterrows():
        pdf_bytes = row['DS_PDF_BYTES']
        file_id = str(row['CD_AVISO_CIRURGIA'])
        
        if pd.isna(pdf_bytes):
            print(f"Skipping row {index}: No PDF bytes available")
            continue
        
        # Extract forms data using Textract
        forms_data = pdf_analyzer.extract_forms_data_from_pdf(
            pdf_bytes=pdf_bytes,
            file_id=file_id
        )
        
        
        # Process with LLM
        forms_data_str = str(forms_data)
        llm_response = llm_instance.invoke_model(input_str=forms_data_str)
        llm_results[file_id] = llm_response
    return llm_results

def llm_dict_to_final_df(llm_results: Dict, df: pd.DataFrame, limit=None) -> pd.DataFrame:
    if limit:
        df = df[:limit]

    llm_results_df = pd.DataFrame.from_dict(llm_results, orient="index")
    llm_results_df.index.name = "CD_AVISO_CIRURGIA"
    llm_results_df = llm_results_df.reset_index()

    # Expand LLM results into separate columns
    llm_expanded_df = pd.json_normalize(llm_results_df.iloc[:, 1:].to_dict('records'))
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_results_df['CD_AVISO_CIRURGIA']

    # Add suffix to distinguish LLM columns
    llm_expanded_df = llm_expanded_df.add_suffix('_llm').rename(columns={'CD_AVISO_CIRURGIA_llm': 'CD_AVISO_CIRURGIA'})

    # Convert CD_AVISO_CIRURGIA to int to match the original dataframe type
    llm_expanded_df['CD_AVISO_CIRURGIA'] = llm_expanded_df['CD_AVISO_CIRURGIA'].astype(int)

    # Inner join with original dataframe
    return df.merge(
        llm_expanded_df, 
        on='CD_AVISO_CIRURGIA', 
        how='inner'
    )



In [27]:
llm_results = llm_extraction(df=final_extracted_df, limit=10)
llm_results_df = llm_dict_to_final_df(llm_results=llm_results, df=final_extracted_df, limit=10)

print(f"Processing completed! Created textract_eval_df with {len(llm_results_df)} rows")
print(f"LLM columns added: {[col for col in llm_results_df.columns if col.endswith('_llm')]}")
llm_results_df.head()

{"timestamp": "2025-08-27T18:15:15", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF uploaded to S3: s3://autorizacoes-textract/guias-pdf/textract_pdfs/857416_20250827_181514_a146ff82.pdf", "filename": "textract_service.py", "lineno": 177}
{"timestamp": "2025-08-27T18:15:16", "level": "INFO", "name": "app.utils.textract_service", "message": "Started PDF analysis with JobId: 83c635feafc1959711a0db312e5cd6822ea03e4476ba4e886366c83f56e60efb", "filename": "textract_service.py", "lineno": 210}
{"timestamp": "2025-08-27T18:15:16", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF analysis in progress for JobId: 83c635feafc1959711a0db312e5cd6822ea03e4476ba4e886366c83f56e60efb", "filename": "textract_service.py", "lineno": 264}
{"timestamp": "2025-08-27T18:15:21", "level": "INFO", "name": "app.utils.textract_service", "message": "PDF analysis in progress for JobId: 83c635feafc1959711a0db312e5cd6822ea03e4476ba4e886366c83f56e60efb", "filename": "textra

Processing completed! Created textract_eval_df with 10 rows
LLM columns added: ['procedimento_autorizado_llm', 'paciente_llm', 'codigo_autorizado_llm', 'senha_llm', 'validade_senha_llm', 'data_solicitacao_llm', 'observacoes_opme_llm', 'observacoes_gerais_llm', 'profissional_solicitante_llm']


,CD_AVISO_CIRURGIA,DS_PROCEDIMENTO,CD_GUIA,CD_SENHA,DS_GUIA_PATH,DS_CONTATO,health_insurance_name,DS_PDF_BYTES,DS_GUIA_PATH_CLICKABLE,procedimento_autorizado_llm,paciente_llm,codigo_autorizado_llm,senha_llm,validade_senha_llm,data_solicitacao_llm,observacoes_opme_llm,observacoes_gerais_llm,profissional_solicitante_llm
0,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE,b'%PDF-1.3\n%\xb7\xbe\xad\xaa\n1 0 obj\n<<\n/T...,"<a href=""https://cdns.overmind.ai/autorizacao-...","EXTENSOS FERIMENTOS, CICATRIZES OU TUMORES EXE...",SALETE EMILIA DOS SANTOS LIMA,30101557 x 1,0001.2025.07-00053801,10/08/25,11/07/25,Pedido sem OPME,Solicitacao de autorizacao,KAYO VIEIRA TEODORAK PÊGO
1,857416,"[{""code"":30101557,""description"":""EXCISAO E ROT...",19494856.0,None,https://cdns.overmind.ai/autorizacao-cemig-000...,\n Procedimento Autorizado\n Pacient...,CEMIG SAUDE,b'%PDF-1.3\n%\xb7\xbe\xad\xaa\n1 0 obj\n<<\n/T...,"<a href=""https://cdns.overmind.ai/autorizacao-...","EXTENSOS FERIMENTOS, CICATRIZES OU TUMORES EXE...",SALETE EMILIA DOS SANTOS LIMA,30101557 x 1,0001.2025.07-00053801,10/08/25,11/07/25,Pedido sem OPME,Solicitacao de autorizacao,KAYO VIEIRA TEODORAK PÊGO
2,857469,"[{""code"":30907136,""description"":""VARIZES - TRA...",19495507.0,J5VEYT7,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...",VARIZES TRATAMENTO CIRURGICO DE DOIS MEMBROS,BARBARA ARAUJO SCHNNEPPEL CORREA,30907136 x 1,J5VEYT7,None,11/07/2025,Pedido sem OPME,Qtde. Diárias: 1,MATEUS ALVES BORGES CRISTINO
3,857883,"[{""code"":30205050,""description"":""AMIGDALECTOMI...",19506292.0,J5VEW29,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Autorizado\n Pacient...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...",AMIGDALECTOMIA DAS PALATINAS,MILENA RAFAELA TRINDADE,"30205050 x 1, 30205069 x 1, 30205271 x 1, 3050...",J5VEW29,None,11/07/2025,Pedido sem OPME,ADM(REDE NACIONAL (0) PL. ADM(REDE NACIONAL (0...,AURELIA ALBUQUERQUE MARTINS
4,858036,"[{""code"":30205247,""description"":""UVULOPALATO-F...",19508906.0,J5VEWF9,https://cdns.overmind.ai/autorizacao-bradesco-...,\n Procedimento Parcialmente autorizado\n...,BRADESCO,b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n4 0 obj\n<</Len...,"<a href=""https://cdns.overmind.ai/autorizacao-...",AMIGDALECTOMIA DAS PALATINAS,DIONE RAIMUNDO CARVALHO PINTO,"30205050 x 1, 30205069 x 1, 30205247 x 1, 3050...",J5VEWF9,None,11/07/2025,Pedido sem OPME,ADM(REDE NACIONAL (0) PL. ADM(REDE NACIONAL (0...,LUCAS EDUARDO DE OLIVEIRA


#### checkpoint

In [28]:
#save textract_eval_df to sql
import sqlalchemy
llm_engine = sqlalchemy.create_engine("sqlite:///dbs/llm_results_df.sqlite")
llm_results_df.to_sql("textract_eval", con=llm_engine, if_exists="replace", index=False)

10

In [29]:
import sqlalchemy
import pandas as pd

llm_engine = sqlalchemy.create_engine("sqlite:///dbs/llm_results_df.sqlite")
llm_results_df = pd.read_sql("SELECT * FROM textract_eval", con=llm_engine)